## Chlorophyll-a forecasting using LSTM Model



#### Getting Started:
1. Before running the notebook, please make sure to have the following python version and libraries are installed <br>
- python 3.9.12
- pytorch (https://pytorch.org/get-started/locally/)

2. Create an account in Weights and Biases (WANDB) (https://wandb.ai/home). While running the notebook, you maybe prompted to enter the WANDB username

<br>
The requirements.txt file lists the basic libraries require. Running the following cell should install all of them (in case they are not already installed). 

In case, any library is missed here, you would be prompted with an ImportError. In such case, just install it with pip (google -> pip install library_name)

In [1]:
!pip install -r requirements.txt

In [2]:
import random
import pandas as pd
import numpy as np
from tqdm import trange
import os
import datetime
import matplotlib.pyplot as plt
import math

import torch
import torch.nn as nn
from torch import optim

from utils import Utils
from encoder_decoder import seq2seq

import warnings
warnings.filterwarnings('ignore')

wandb: Currently logged in as: rladwig (computational-limnology). Use `wandb login --relogin` to force relogin


## 0. GPU Selection
Check if GPU is available on the machine the notebook is running. If yes, then assign a GPU, else run it on CPU

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device)

cuda


## 1. Parameter setting

#### Specify the wandb project and wandb run
wandb refers to Weights and Biases. Integrating this tool into the notebook will allow it to access the run details and generate train and test curves, among many other information

In [4]:
# wandb project name
wandb_project = "mcl_lstm"

# wandb run name
wandb_run = "test_run_{}_{}".format(str(datetime.datetime.now().date()), str(datetime.datetime.now().time()))

# Yes if we want wandb to save our python code, else no
save_code = True

#### Specify the input path (where the dataset is stored) and dataset name
Note: For different dataset, the processing/handling can/will be different. In this notebook, FCR (observational) data has been considered. It also has a metadata file that stores the column names and types. 
<br>
For the purpose of the tutorial, the notebook is kept simple, hence, going with FCR data for now

In [5]:
# Input path
path = './'

# Name of the file
file = '../1_trainingData/COMBINED-all_data_lake_modeling_in_time.csv'

# Name of the metadata file
#../metadata = 'LSTM_dataset_column_key_07OCT22.csv'

#### Specify the Time-series specific parameters

In [6]:
# Lookback window
input_window = 6

# horizon window
output_window = 6

# stride - While creating samples (lookback window + horizon window = 1 sample) define the amount of stride the sliding window needs to take
stride = 1

# The ratio in which train and test data is split. If it is 0.8, then first 80% of data goes into train and remaining 20% into test
split_ratio = 0.6

#### Specify the model specific parameters

In [7]:
# Types of Model include: LSTM, GRU, RNN
model_type = "LSTM"

# Number of layers in our deep learning model
num_layers = 2

# Hidden cell (RNN/LSTM/GRU) size
hidden_feature_size = 32

# Output size of our encoder_decoder model, i.e. number of target variables
output_size = 1

'''
Model Training parameters
'''
# batch_size during training
batch_size = 4#32

# Number of epochs we want to train the model for (1 epoch = 1 pass of the complete training data through the model)
epochs = 100

# Learning rate specifies the rate at which we want to update the model parameters after every training pass
learning_rate = 0.001

# Eval freq says how frequently during training do you want to evaluate your model on the validation data (to see its performance on non-training data)
eval_freq = 1 # logic is -> if iteration_num % eval_freq == 0 -> then perform evaluation

# While generating the training batches do we want the generator to shuffle the batches?
batch_shuffle = True

# Dropout is a form of regularization
dropout = 0.0

'''
Learning rate scheduler parameters
'''
max_lr=5e-3
div_factor=100
pct_start=0.05 
anneal_strategy='cos'
final_div_factor=10000.0

'''
Parameters for early stopping
'''
# Set to True if we want Early stopping
early_stop = False

# If there is no improvement for a 'thres' number of epocs stop the training process
thres=5

# Quantifying the improvement. If the validation loss is greater than min_val_loss_so_far + delta for thres number of iterations stop the training
delta=0.5

'''
Other parameters
'''
# Specify the amount of L2 regularization to be applied.
weight_decay=0.0

# Specify the percentage of times we want to enforce teacher forcing
teacher_forcing_ratio = 0.0
training_prediction = 'recursive'

## 2. Data Processing

#### Read the metadata file

In [8]:
depth_steps = 25 * 2 

depth_list = np.array(list(range(1, depth_steps+1))   )*0.5



In [9]:
#incoming_temp = ['temp_initial00_{}'.format(x) for x in depth_list]
#outgoing_temp = ['temp_heat01_{}'.format(x) for x in depth_list]

incoming_temp = ['temp_initial00']
outgoing_temp = ['temp_heat01']

#dx = pd.read_csv(os.path.join(path,file))

#feature_cols = ['AirTemp_degC', 'Longwave_Wm-2', 'Latent_Wm-2', 'Sensible_Wm-2', 'Shortwave_Wm-2',
#                'lightExtinct_m-1', 'ShearStress_Nm-2',
#                 'day_of_year', 'time_of_day', 'ice', 'snow', 'snowice','Volume_m2','Osgood','MaxDepth_m',
#                'MeanDepth_m','Area_m2'] + incoming_temp

feature_cols = ['AirTemp_degC', 'Longwave_Wm-2', 'Latent_Wm-2', 'Sensible_Wm-2', 'Shortwave_Wm-2',
                'lightExtinct_m-1', 'ShearStress_Nm-2',
                 'day_of_year', 'time_of_day', 'ice', 'snow', 'snowice'] + incoming_temp

#feature_cols = ['AirTemp_degC','day_of_year', 'time_of_day'] + incoming_temp

date_col = ['time']

target_cols = outgoing_temp

In [10]:
def cycle_encode(x, period):
    sin = np.sin(2*math.pi*x/period)
    cos = np.cos(2*math.pi*x/period)
    
    return sin, cos

In [11]:
feature_cols.remove('day_of_year')
feature_cols.remove('time_of_day')
feature_cols

['AirTemp_degC',
 'Longwave_Wm-2',
 'Latent_Wm-2',
 'Sensible_Wm-2',
 'Shortwave_Wm-2',
 'lightExtinct_m-1',
 'ShearStress_Nm-2',
 'ice',
 'snow',
 'snowice',
 'temp_initial00']

In [12]:
feature_cols += ['doy_sin', 'doy_cos', 'tod_sin', 'tod_cos']

In [13]:
#dx = pd.read_csv(os.path.join(path, metadata))

# Extract all col names from Metadata
#feature_cols = dx[dx['column_type']=='feature']['column_names'].tolist()  # feature colums represent the input drivers
#target_cols = dx[dx['column_type']=='target']['column_names'].tolist()   # target column represent the chlorophyll values
#date_col = dx[dx['column_type']=='date']['column_names'].tolist()[0]    # date column stores the date timeline

In [14]:
# Specify whether we want to add chlorophyll to the input feature set
#feature_cols += target_cols
feature_cols

['AirTemp_degC',
 'Longwave_Wm-2',
 'Latent_Wm-2',
 'Sensible_Wm-2',
 'Shortwave_Wm-2',
 'lightExtinct_m-1',
 'ShearStress_Nm-2',
 'ice',
 'snow',
 'snowice',
 'temp_initial00',
 'doy_sin',
 'doy_cos',
 'tod_sin',
 'tod_cos']

In [15]:
target_cols

['temp_heat01']

#### Create an utility object
An object of the Utils class, it contains all the utility functions like splitting train and test data, normalizing the data, etc.

In [16]:
'''
Utility instance - to perform data processing, train test split
'''
utils = Utils(num_features=len(feature_cols), inp_cols=feature_cols, target_cols=target_cols, date_col=date_col,
              input_window=input_window, output_window=output_window, num_out_features=output_size, stride=stride)

#### Read the dataset

In [17]:
'''
Read data
'''
df = pd.read_csv(path+file)

In [18]:
doy_sin, doy_cos = cycle_encode(df.day_of_year.values, 365)

tod_sin, tod_cos = cycle_encode(df.time_of_day.values, 24)

In [19]:
df['doy_sin'] = doy_sin
df['doy_cos'] = doy_cos


In [20]:
df['tod_sin'] = tod_sin
df['tod_cos'] = tod_cos


In [21]:
print(df.shape)


(5781485, 55)


In [22]:
df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_initial00,obs_temp,input_obs,ice,snow,snowice,doy_sin,doy_cos,tod_sin,tod_cos
0,1.0,2017-06-28 21:00:00,13.234292,-98.716740,-54.502190,-10.905886,11.655360,0.63,199470.305447,0.001763,...,16.786980,16.707840,16.810400,0.0,0.0,0.0,0.060213,-0.998186,-0.707107,0.707107
1,2.0,2017-06-28 21:00:00,13.234292,-98.716740,-54.502190,-10.905886,11.655360,0.63,199470.305447,0.001763,...,16.786980,16.712420,16.814190,0.0,0.0,0.0,0.060213,-0.998186,-0.707107,0.707107
2,3.0,2017-06-28 21:00:00,13.234292,-98.716740,-54.502190,-10.905886,11.655360,0.63,199470.305447,0.001763,...,16.786980,16.733420,16.833630,0.0,0.0,0.0,0.060213,-0.998186,-0.707107,0.707107
3,4.0,2017-06-28 21:00:00,13.234292,-98.716740,-54.502190,-10.905886,11.655360,0.63,199470.305447,0.001763,...,16.786980,16.742480,16.840190,0.0,0.0,0.0,0.060213,-0.998186,-0.707107,0.707107
4,5.0,2017-06-28 21:00:00,13.234292,-98.716740,-54.502190,-10.905886,11.655360,0.63,199470.305447,0.001763,...,16.737530,16.638270,16.735570,0.0,0.0,0.0,0.060213,-0.998186,-0.707107,0.707107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5781480,5.0,2023-07-04 14:00:00,23.289200,-59.086913,-2.191761,0.511323,544.901536,1.30,4.861124,0.000466,...,8.125515,11.129420,11.186380,0.0,0.0,0.0,-0.043022,-0.999074,-0.500000,-0.866025
5781481,6.0,2023-07-04 14:00:00,23.289200,-59.086913,-2.191761,0.511323,544.901536,1.30,4.861124,0.000466,...,6.039822,10.854090,10.858545,0.0,0.0,0.0,-0.043022,-0.999074,-0.500000,-0.866025
5781482,7.0,2023-07-04 14:00:00,23.289200,-59.086913,-2.191761,0.511323,544.901536,1.30,4.861124,0.000466,...,5.127928,10.872310,10.870905,0.0,0.0,0.0,-0.043022,-0.999074,-0.500000,-0.866025
5781483,8.0,2023-07-04 14:00:00,23.289200,-59.086913,-2.191761,0.511323,544.901536,1.30,4.861124,0.000466,...,4.839974,10.867625,10.867030,0.0,0.0,0.0,-0.043022,-0.999074,-0.500000,-0.866025


In [23]:
uniquelakes = df['ID'].unique()
print(uniquelakes)


['ERK' 'RBR' 'FCR']


In [24]:
Xtrain = []
Ytrain = []
Xtest = []
Ytest = []

In [25]:
run = 0
for uniquelakes_id in uniquelakes:
    lake_df = df[df['ID'] == uniquelakes_id]
    
    unique_depth = lake_df['depth'].unique()
    
    for unique_depth_id in unique_depth:
        depth_df = lake_df[lake_df['depth'] == unique_depth_id]
        
        df_depth_train, df_depth_test = utils.train_test_split(depth_df, split_ratio=split_ratio)
        
        Xtrain_depth, Ytrain_depth = utils.windowed_dataset(df_depth_train)
        Xtest_depth, Ytest_depth = utils.windowed_dataset(df_depth_test)
        
        if run == 0:
            Xtrain = Xtrain_depth
            Ytrain = Ytrain_depth
            Xtest = Xtest_depth
            Ytest = Ytest_depth
            
            run = run+1

        else:
            Xtrain = np.concatenate([Xtrain, Xtrain_depth], axis = 0)
            Ytrain = np.concatenate([Ytrain, Ytrain_depth], axis = 0)
            Xtest = np.concatenate([Xtest, Xtest_depth], axis = 0)
            Ytest = np.concatenate([Ytest, Ytest_depth], axis = 0)
        

In [26]:
Xtrain.shape

(3467580, 6, 15)

In [27]:
Xtrain_depth.shape

(26268, 6, 15)

In [28]:
Xtrain = utils.normalize_numpy(Xtrain, feat_or_target="feat")
Ytrain = utils.normalize_numpy(Ytrain, feat_or_target="target")
Xtest = utils.normalize_numpy(Xtest, feat_or_target="feat", use_stat=True)
Ytest = utils.normalize_numpy(Ytest, feat_or_target="target", use_stat=True)

In [29]:
# normalize
# train
Xtrain

array([[[ 6.59321266e-01, -2.18873012e+00, -1.15854605e+00, ...,
         -1.49973811e+00, -9.99937328e-01,  9.99751635e-01],
        [ 3.47803554e-01, -2.30813268e+00, -6.63563963e-01, ...,
         -1.49973811e+00, -7.07044109e-01,  1.22449651e+00],
        [ 1.15626717e-01, -2.47768103e+00, -1.48840529e+00, ...,
         -1.49973811e+00, -3.65962731e-01,  1.36577705e+00],
        [-2.84774448e-04, -2.52824953e+00, -8.00501914e-01, ...,
         -1.50100313e+00,  6.26735280e-05,  1.41396521e+00],
        [-1.15767683e-01, -2.65817796e+00, -7.63022800e-01, ...,
         -1.50100313e+00,  3.66088078e-01,  1.36577705e+00],
        [-2.06680877e-01, -2.74453503e+00, -9.02385621e-01, ...,
         -1.50100313e+00,  7.07169456e-01,  1.22449651e+00]],

       [[ 3.47803554e-01, -2.30813268e+00, -6.63563963e-01, ...,
         -1.49973811e+00, -7.07044109e-01,  1.22449651e+00],
        [ 1.15626717e-01, -2.47768103e+00, -1.48840529e+00, ...,
         -1.49973811e+00, -3.65962731e-01,  1.36577

#### Train Test split
Ideally, a 3-way split is done - train, val and test. The validation split is generally used to tune the hyper-parameters during training. Once the hyper-parameters are tuned, the model
is re-trained on the train+val data. To keep the notebook short and simple, hyper-parameter tuning is not included

#### Normalize the data
Standard normalization - 0 mean and 1 standard deviation

In [30]:
utils.inp_cols

['AirTemp_degC',
 'Longwave_Wm-2',
 'Latent_Wm-2',
 'Sensible_Wm-2',
 'Shortwave_Wm-2',
 'lightExtinct_m-1',
 'ShearStress_Nm-2',
 'ice',
 'snow',
 'snowice',
 'temp_initial00',
 'doy_sin',
 'doy_cos',
 'tod_sin',
 'tod_cos']

In [31]:
utils.inp_cols

['AirTemp_degC',
 'Longwave_Wm-2',
 'Latent_Wm-2',
 'Sensible_Wm-2',
 'Shortwave_Wm-2',
 'lightExtinct_m-1',
 'ShearStress_Nm-2',
 'ice',
 'snow',
 'snowice',
 'temp_initial00',
 'doy_sin',
 'doy_cos',
 'tod_sin',
 'tod_cos']

In [32]:
utils.target_cols

['temp_heat01']

In [33]:
'''
convert the mean and std to torch
'''
utils.y_mean = torch.tensor(utils.y_mean, device=device)
utils.y_std = torch.tensor(utils.y_std, device=device)

In [34]:
utils.y_mean

tensor([5.1218], device='cuda:0', dtype=torch.float64)

In [35]:
utils.y_std

tensor([4.4593], device='cuda:0', dtype=torch.float64)

In [36]:
utils.num_features = len(utils.inp_cols)

#### Create train and test samples
Each sample is created using a sliding window. 1 sliding window = 1 lookback window + 1 horizon window = 1 sample

In [37]:
Xtrain.shape

(3467580, 6, 15)

In [38]:
Ytrain.shape

(3467580, 6, 1)

In [39]:
Xtest.shape

(2311375, 6, 15)

In [40]:
Ytest.shape

(2311375, 6, 1)

In [41]:
Ytrain[[2]]

array([[[2.56587462],
        [2.56184568],
        [2.55969853],
        [2.54836956],
        [2.53568805],
        [2.53693483]]])

#### Datatype conversion to torch

In [42]:
'''
Convert data into torch type
'''
X_train, Y_train, X_test, Y_test = utils.numpy_to_torch(Xtrain, Ytrain, Xtest, Ytest)

In [43]:
del df

## 3. Modeling

#### Define the model

In [44]:
'''
Create the seq2seq model
'''
model = seq2seq(input_size = X_train.shape[2], 
                hidden_size = hidden_feature_size, 
                output_size=output_size,
                model_type=model_type,
                num_layers = num_layers,
                utils=utils,
                dropout=dropout,
                device=device
               )

#### Train the model

In [45]:
'''
Train the model
'''
config = {
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": learning_rate,
    "eval_freq": eval_freq,
    "batch_shuffle": batch_shuffle,
    "dropout":dropout,
    "num_layers": num_layers,
    "hidden_feature_size": hidden_feature_size,
    "model_type": model_type,
    "teacher_forcing_ratio": teacher_forcing_ratio,
    "max_lr": max_lr,
    "div_factor": div_factor,
    "pct_start": pct_start,
    "anneal_strategy": anneal_strategy,
    "final_div_factor": final_div_factor,
    "dataset": file,
    "split_ratio":split_ratio,
    "input_window":input_window,
    "output_window":output_window,
    "early_stop_thres":thres,
    "early_stop_delta":delta,
    "early_stop":early_stop,
    "weight_decay":weight_decay
}
loss, test_rmse, train_rmse = model.train_model(X_train, 
                                                Y_train,
                                                X_test,
                                                Y_test,
                                                target_len = output_window,
                                                config = config,
                                                training_prediction = training_prediction,  
                                                dynamic_tf = False,
                                                project_name = wandb_project,
                                                run_name = wandb_run,
                                                save_code = save_code)

  0%|                                                                  | 877/866895 [00:07<2:01:38, 118.65it/s]


  0%|▏                                                                | 1783/866895 [00:15<2:03:14, 116.99it/s]


  0%|▏                                                                | 2698/866895 [00:23<2:01:51, 118.20it/s]


  0%|▎                                                                | 3611/866895 [00:30<2:01:35, 118.33it/s]


  1%|▎                                                                | 4531/866895 [00:38<1:59:37, 120.15it/s]


...

100%|██████████████████████████████████████████████████████████████▊| 864206/866895 [2:02:08<00:22, 118.50it/s]


100%|██████████████████████████████████████████████████████████████▊| 865116/866895 [2:02:16<00:14, 120.07it/s]


100%|██████████████████████████████████████████████████████████████▉| 866037/866895 [2:02:24<00:07, 117.54it/s]


  0%|                                                                                | 0/100 [2:02:32<?, ?it/s]


RuntimeError: Input and parameter tensors are not at the same device, found input tensor at cpu and parameter tensor at cuda:0

In [ ]:
plt.figure(figsize=(5,4), dpi=150)
plt.plot(train_rmse, lw=2.0, label='train_rmse')
plt.plot(test_rmse, lw=2.0, label='test_rmse')
plt.yscale("log")
plt.grid("on", alpha=0.2)
plt.legend()
plt.show()

#### Plot the train test curves

In [ ]:
plt.figure(figsize=(5,4), dpi=150)
plt.plot(train_rmse, lw=2.0, label='train_rmse')
plt.plot(test_rmse, lw=2.0, label='test_rmse')
plt.yscale("log")
plt.grid("on", alpha=0.2)
plt.legend()
plt.show()

#### Save the model

In [ ]:
load = False

# If load=True, specify the model to load in the below line
MODEL_PATH = "./models/model_weights_test_run_2023-03-21_22:42:45.577525"

In [ ]:

if load:
    model.load_state_dict(torch.load(MODEL_PATH))
else:
    MODEL_PATH='./models/model_weights_{}'.format(wandb_run[:22])
    if not os.path.exists('./models'):
        os.mkdir('./models')
    torch.save(model.state_dict(), MODEL_PATH)

In [ ]:
'''
Perform evaluation
'''
train_eval_dict = model.evaluate_batch(X_train.to(device), Y_train.to(device))
test_eval_dict = model.evaluate_batch(X_test.to(device), Y_test.to(device))

In [ ]:
X_test.shape

In [ ]:
X_train.shape

In [ ]:
Y_train.shape

In [ ]:
Y_test.shape

In [ ]:
test_eval_dict['y_true'][1001][2]

In [ ]:
test_eval_dict['y_pred'][1001][2]

## 4. Plotting and Evaluation

In [ ]:
'''
Create plot tables for T+n th predictions
'''
train_gt = train_eval_dict['y_true']
train_gt_df = pd.DataFrame(train_gt.cpu().numpy()[:,:,0])
train_gt_values = np.append(train_gt_df[0].values, train_gt_df.iloc[-1,1:]) # ground-truth values for train data

test_gt = test_eval_dict['y_true']
test_gt_df = pd.DataFrame(test_gt.cpu().numpy()[:,:,0])
test_gt_values = np.append(test_gt_df[0].values, test_gt_df.iloc[-1,1:]) # ground-truth values for test data

train_pred = train_eval_dict['y_pred'] # model predicted values for train data
test_pred = test_eval_dict['y_pred'] # model predicted values for test data

df_train_comp = df_train
#df_train_comp=df_train_comp.rename(columns = {'time':'Date'})
#print(df_train_comp.Date)

df_test_comp = df_test
#df_test_comp=df_test_comp.rename(columns = {'time':'Date'})
#print(df_test_comp.Date)

print(df_train_comp.shape)
print(train_pred.shape)
print(train_gt_values.shape)

train_T_pred_table, train_plot_df, plot_train_gt_values = utils.predictionTable(df_train_comp, train_pred, train_gt_values)

test_T_pred_table, test_plot_df, plot_test_gt_values = utils.predictionTable(df_test_comp, test_pred, test_gt_values)

In [ ]:
'''
Generate the plots on train data
'''

# Specify the list of T+n predictions to plot
horizon_range = [1,7, 14] # this will plot T+1 and T+n predictions w.r.t Ground truth

utils.plotTable(train_plot_df, plot_train_gt_values, horizon_range)

In [ ]:
'''
Generate the plots on test data
'''

# Specify the list of T+n predictions to plot
horizon_range = [1,7,14] # this will plot T+1 and T+n predictions w.r.t Ground truth

utils.plotTable(test_plot_df, plot_test_gt_values, horizon_range)

#### Compute the RMSE values

In [ ]:
'''
Compute train rmse

- Train RMSE values for all T+n th predictions. The index represents the T+n

'''
rmse_values = []
for i in range(output_window):
    rmse_values.append(utils.compute_rmse(i, train_T_pred_table, train_gt_values))
rmse_values = pd.DataFrame(rmse_values, columns=['RMSE'], index=range(1,output_window+1))
rmse_values

In [ ]:
'''
Compute test rmse

- Test RMSE values for all T+n th predictions. The index represents the T+n

'''
test_rmse_values = []
for i in range(output_window):
    test_rmse_values.append(utils.compute_rmse(i, test_T_pred_table, test_gt_values))
test_rmse_values = pd.DataFrame(test_rmse_values, columns=['RMSE'], index=range(1,output_window+1))
test_rmse_values